# 🎯 Where's Wally AI Challenge — Student Notebook

Welcome to the **Wally AI Challenge**!  
Your goal today is to explore how AI finds **Wally** in images and try to "break" the model using different tricks.

You will:
- Detect Wally using AI  
- Resize images to see when the AI fails  
- Try occlusion attacks  
- Rank image confidence  
- Earn points for your team  

Let’s begin!

---------------------------------------
## 📝 Submit Your Team Name

**Click the link below to submit your team name on Slido:**

[Open Slido](https://app.sli.do/event/aKKyB47ajZ7yERZN35rEgg/embed/polls/d67f5aee-ef9f-4574-ad8b-0d08bac98eae)


----------------------------------------------
## 📦 Load the Wally AI Tools

These utilities give you:
- `detect_wally()` → show where the AI thinks Wally is  
- `scale_guess()` → shrink the image (3 guesses only!)  
- `blur_attack()` → blur Wally and test robustness  
- `augmented_versions()` → generate brightness/contrast/flip versions  

Just import them below.

**To run a cell click inside the cell and press `CTRL` and `ENTER` at the same time**

In [ ]:
!pip install ultralytics pillow matplotlib numpy opencv-python

In [ ]:
from ultralytics import YOLO
import numpy as np
from PIL import Image, ImageEnhance, ImageOps
import matplotlib.pyplot as plt
import cv2
import os
from utilities import (
    detect_wally, show,
    scale_guess, reset_scale_game, calculate_scale_final_score, scale_challenge_complete,
    blur_attack, blur_challenge_complete, augmented_versions,
    display_images_for_ranking, evaluate_ranking_predictions, waldo_sizing_challenge
)

print("Utilities loaded! Ready to start the challenge.")

# 🟦 Task 0 — Basic Detection (Warm-Up)

This confirms everything is working correctly.

Press `CTRL` and `ENTER` at the same time


In [ ]:
test_img = "test_waldo_image.jpg"   # replace with your actual test file
dets, annotated, _ = detect_wally(test_img)
show(annotated, "Detected Wally")

----------------
## 📐Task 1 — Scale until you Fail Challenge

Your mission: **Find the MINIMAL scale that breaks the AI - precision matters!**

### 📏 What does scaling mean?

**Scaling shrinks the whole image** - like zooming out on your phone:

- **Scale 1.0** = Original size (100%)
- **Scale 0.5** = Half size (50%) 
- **Scale 0.32** = Sweet spot (32% - right at the edge!)
- **Scale 0.1** = Tiny image (10% - overkill!)

**Everything gets smaller together** - the crowd, buildings, AND Wally!

### 🎮 The Challenge:
Find the **smallest scale** that still breaks the AI without going overboard!

- You get **3 guesses only**
- **Precision is key** - going too small loses points!
- **Important:** Enter values as `0.4,0.32,0.28` for example (no spaces after commas!)

### 🏆 Scoring:
- **Perfect precision** (optimal breaking point) → **10 points**
- **Close to optimal** (near the sweet spot) → **8 points**
- **Good break** (effective but not precise) → **6 points**
- **Overkill break** (way too small - too easy!) → **4 points**
- **No break achieved** (AI survives) → **2 points**

**Goal:** Break the AI with the **minimum necessary scaling** - show your precision!

### 🏁 To Start:
- Run the cell below by pressing `CTRL` and `ENTER` at the same time
- A box will appear at the top for you to input your answer

In [ ]:
user_input = input("Please input three scale guesses (comma-separated, e.g. 0.5,0.3,0.1): ")

# Parse the input
guesses = [float(x.strip()) for x in user_input.split(',')]
attempt_scores = []

# Test each guess and collect scores
for i, scale_val in enumerate(guesses, 1):
    print(f"\n--- Guess {i}: scale = {scale_val} ---")
    dets, ann, msg, points = scale_guess(test_img, scale_val)
    attempt_scores.append(points)
    show(ann, f"{msg}")

final_score = scale_challenge_complete(attempt_scores)

----------------------------

# 🎭 Task 2 — Break the AI with Blur Attack

Your mission is to **break the AI** by blurring the detection area (Wally).

You'll try **3 different blur strengths** to see when the AI can no longer detect Wally.

### Rules
- You get **3 attempts** with different blur strengths
- Each blur strength must be different
- **Must be odd numbers** (the code will fix it if you forget!)

### 🏆 Scoring:
- **Perfect precision** → **10 pts**
- **Very close** (within 2 of perfect) → **8 pts**
- **Close** (within 5 of perfect) → **6 pts**
- **Break but not optimal** → **4 pts**
- **AI survives** → **2 pts**

**Goal:** Find the **minimum blur** that breaks the AI - going too high loses precision points!

**Tip:** Start with a mild blur (e.g., 5), then increase difficulty (e.g. 10).

### 🏁 To Start:
- Run the cells below by pressing `CTRL` and `ENTER` at the same time
- A box will appear at the top for you to input your answer


In [ ]:
# Store all blur attempts and scores
blur_attempts = []
blur_scores = []

blur_val_1 = int(input("Enter your FIRST blur strength (try a small odd number like 15): "))
print(f"\n--- Attempt 1: blur strength = {blur_val_1} ---")
dets_1, ann_1, msg_1, points_1 = blur_attack(test_img, blur_strength=blur_val_1)
blur_attempts.append({'blur': blur_val_1, 'points': points_1, 'detected': len(dets_1) > 0})
blur_scores.append(points_1)
show(ann_1, f"Attempt 1: {msg_1}")

In [ ]:
blur_val_2 = int(input("Enter your SECOND blur strength: "))
print(f"\n--- Attempt 2: blur strength = {blur_val_2} ---")
dets_2, ann_2, msg_2, points_2 = blur_attack(test_img, blur_strength=blur_val_2)
blur_attempts.append({'blur': blur_val_2, 'points': points_2, 'detected': len(dets_2) > 0})
blur_scores.append(points_2)
show(ann_2, f"Attempt 2: {msg_2}")

In [ ]:
blur_val_3 = int(input("Enter your THIRD blur strength: "))
print(f"\n--- Attempt 3: blur strength = {blur_val_3} ---")
dets_3, ann_3, msg_3, points_3 = blur_attack(test_img, blur_strength=blur_val_3)
blur_attempts.append({'blur': blur_val_3, 'points': points_3, 'detected': len(dets_3) > 0})
blur_scores.append(points_3)
show(ann_3, f"Attempt 3: {msg_3}")

final_blur_score = blur_challenge_complete(blur_scores, blur_attempts)

--------------------------------------
# 🎯 Task 3 — Waldo Sizing Challenge

Your mission: **Find the smallest Waldo size that the AI can still detect with confidence!**

### How it works:
1. **You get 3 attempts** to find the optimal Wally size
2. **Enter a height value** (10-300 pixels) - width is calculated automatically (typing 300 -> 300 pixels)
3. **The AI tests how well it can detect** your custom-sized Wally
4. **Earn points** based on how small you can make Wally while keeping the AI confident!

### 🏆 Scoring System:
- **Tiny Wally Detected** → **15 pts** (Legendary!)
- **Small Wally Detected** → **12 pts** (Incredible - at the limit!)
- **Little Wally Detected** → **10 pts** (Excellent!)
- **Medium Sized Wally Detected** → **8 pts** (Great!)
- **Big Wally Detected** → **6 pts** (Good!)
- **Large Wally Detected** → **4 pts** (Okay!)
- **XL Wally Detected** → **2 pts** (Too easy!)

### 🎯 Bonus Points:
- **95%+ confidence** → **+2 bonus pts**
- **90%+ confidence** → **+1 bonus pt**

**Remember:** You're trying to find the **smallest size** that still works reliably!

### 🏁 To Start:
- Run the cell below by pressing `CTRL` and `ENTER` at the same time to see Wally and the background separately


In [ ]:
# Load and display images
waldo_sprite = Image.open('images/finding_waldo/waldo.png').convert("RGBA")
bg_full = Image.open('images/finding_waldo/wheres_wally.jpg').convert("RGB")

# Print Waldo's original size for reference
waldo_width, waldo_height = waldo_sprite.size
print(f"📏 Reference: Original Waldo size is {waldo_width}x{waldo_height} pixels")
print(f"   You will be changing the second number (height). Use this as a baseline for your sizing attempts!")
print()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(waldo_sprite), ax1.set_title(f"Wally ({waldo_width}x{waldo_height}px)"), ax1.axis('off')
ax2.imshow(bg_full), ax2.set_title("Background"), ax2.axis('off')
plt.show()

### ⏭️ Next:
- Run the cell below and submit your sizes in the box that appears at the top

In [ ]:
# Run the challenge
final_score, attempt_results = waldo_sizing_challenge(200, 150, waldo_sprite, bg_full)

-----------------------------------------------
# 📊 Task 4 — Confidence Prediction Challenge

Your mission: **Predict how confident the AI will be** when detecting Wally in different images!

### How it works:
1. **Study the images** carefully (they'll be shown without AI detection)
2. **Predict the confidence score** (0-100%) for each image
3. **Run the AI** to see the actual confidence scores
4. **Earn points** based on how close your predictions are!

### 🏆 Scoring (per image):
- Within 5% of actual confidence → **10 pts**
- Within 10% → **7 pts**
- Within 15% → **5 pts**
- Within 20% → **3 pts**
- More than 20% off → **0 pts**

**Strategy:** Look for factors like:
- How cluttered is the scene?
- How small is Wally?
- Are there similar-looking objects?
- Is Wally partially hidden?

### 🏁 Part 1: Make Your Predictions
- Input your prediction in the bo at the top. e.g. 80% confident should be written as 80

In [ ]:
image_list = [
    "wally_test4.jpg",
    "images/finding_waldo/testing_waldo_sde.jpg",
    "wally_test5.jpg",
    "wally_test6.jpg",
]


predicted_confidences = display_images_for_ranking(image_list)

### 💯 Part 2: See the Results!

- Run this cell to see how the AI actually performed and calculate your score.

In [ ]:
# Task 4 - Part 2: Evaluate predictions
points = evaluate_ranking_predictions(image_list, predicted_confidences)

----------------------------
# 🧩 Task 5 — AI Failure Analysis

Look at the challenge image below. The AI may struggle to detect Wally correctly.

Your task: **Explain WHY the AI might fail or perform poorly on this image.**

### 🤔 Consider these factors:
- **Clutter**: Are there too many similar-looking objects or distracting patterns?
- **Scale**: Is Wally too small or at an unusual size?
- **Occlusion**: Is Wally partially hidden behind objects?
- **Image quality**: Is the image blurry, low-resolution, or poorly lit?
- **Similar features**: Are there other objects with red/white stripes or similar colors?

### 🏆 Scoring:
- Detailed, insightful explanation with specific observations → **10 pts**
- Good explanation covering main factors → **7 pts**
- Basic explanation with some valid points → **5 pts**
- Vague or minimal explanation → **3 pts**

**Tip:** Look carefully at the image and think like the AI - what visual features make this hard?

In [ ]:
hard = "wallytest3.jpg"  
dets, ann, _ = detect_wally(hard)
show(ann, "Challenge Image")

### 📝 Submit Your Answer

**Click the link below to submit your analysis on Slido:**

[Open Slido](https://app.sli.do/event/aKKyB47ajZ7yERZN35rEgg/embed/polls/cb6303fd-01c6-4077-973e-c22ee7459eae)

Explain why the AI might struggle with this image and earn up to **10 points** for your team!

---------------------
## ✅ MISSION COMPLETE!!

- make sure you have submitted all your score tokens to the front 